<a href="https://colab.research.google.com/github/danielpaz88/FlyRank-task-1/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielpaz88/FlyRank-task-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My temporary lane is "Refresh / Content Opportunity Scoring" (Lane 2). I choose this lane because the goal is to predict ehich web sites should be revised first to update, and this aligns directly to the is_declining_label tag which I explored in notebook 02. Furthermore, the data contain observable signals such as impressions_90d, avg_position, ctr and engagement_rate which can anticipate traffics losses. This lane allows me to build a ranking with reason codes, being this the most useful output for a content editor.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. The question: decision, action, cost of a wrong call

Decision for improvement: Which sites should I send to the refresh queue?

who acts?: An editor or content manager is who revises the generated ranking by the model and prioritizes the sites with highes scores to rewrite or refresh them.

Error corst:

False positive: The editor looses time checking content which doesn't need to. Cost: waste labour hours.

False Negative: The site continues loosing visits and eranings. Cost: lost traffic and earnings oportunities lost.

Why ML?: As we saw in notebook 02, a hand written rule (stale x visible) reaches Precision@50 = 0.680, but a decision tree can reach 0.720 (max_dept=4). This shows that ML finds signal combinations that a rule doesn't.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

pass

## 3. Quick look at the data (2-3 real numbers)

The dataset contains 30,000 anonymized pages. The rate of pages in decline (is_declining_label) is 54.2 %.

If we apply the rule (stale and visible) we obtain Precision@50 of 0.680, which is that 68 % of the first 50 recommended pages are really declinining.

Incorporating engagement_rate and augmenting the depth to 4, the model improved to 0.720 surpassing the hand rule. This could tranlate into time saving for the staff, showing this lane has potential.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

# 1. Declining rate
decline_rate = df["is_declining_label"].mean()
print(f"1. Declining pages rate: {decline_rate:.1%}")

# 2 y 3. hand rule vs. decision tree (replicando el notebook 02)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Hand rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]
y = df["is_declining_label"].values
p50_hand = precision_at_k(df["hand_rule_score"], y, 50)
print(f"2. Hand rule precision (Top 50): {p50_hand:.3f}")

# Tree with engagement_rate
from sklearn.tree import DecisionTreeClassifier
features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]
p50_tree = precision_at_k(tree_score, y, 50)
print(f"3. Decision tree precision (Top 50): {p50_tree:.3f}")
print(f"   ML improvement over manual rules: {p50_tree - p50_hand:.3f}")

30000 pages |  declining rate: 0.542
1. Declining pages rate: 54.2%
2. Hand rule precision (Top 50): 0.680
3. Decision tree precision (Top 50): 0.720
   ML improvement over manual rules: 0.040


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.